# Lanka Energy: implementation walkthrough

Real RECON-SL records, next-month consumption in kWh, and a historical LECO-area demonstration. Use Python 3.12 with requirements-dev.txt installed. The cells call the shared executable pipeline.

In [ ]:
from pathlib import Path
import os, sys
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

## 1. Understand the data and prevent future information

Six previous monthly readings form the history. Household information is masked until the survey date. Targets and missing months are never filled using future observations.

In [ ]:
from energy.features import build_panel
panel = build_panel()
panel.head()

## 2. Train, select, calibrate, then test

Entire households are reserved before fitting. Validation chooses models and blend weights. July calibrates intervals; August–October measures the frozen model. The next command reproduces the experiment and overwrites its generated artifacts.

In [ ]:
from energy.train import train
summary = train()
summary['test_results']

## 3. Read the evidence

MAE is measured in kWh, not an accuracy percentage. Compare against repeating the last reading and check interval coverage separately. The first run showed a small mean-error improvement that must be reported honestly.

In [ ]:
import json
results = json.loads((root / 'results/metrics.json').read_text())
results['context_ablation_validation']

## 4. Load the actual exported model

Serving loads the trained artifact. Input history is oldest to newest. Explanations describe model associations, not measured appliance energy use.

In [ ]:
from energy.predict import Forecaster
demo = json.loads((root / 'models/examples.json').read_text())[0]
Forecaster().predict(demo['history'], demo['forecast_month'], demo['profile'], demo['consumption_limit'])

## 5. Test and deploy

Run `python -m pytest -q`, then launch the API using README.md. Verify the dashboard, `/health` and a forecast on Azure before claiming deployment success. Report and slides should explain the measured outcomes and their limits.